In [ ]:
!pip install tqdm

import nltk as nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from tqdm import tqdm
from collections import defaultdict, Counter
import numpy as np
import math as math

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

stop_words = set(stopwords.words('english'))
df = pd.DataFrame(pd.read_json('/content/drive/MyDrive/Information_Retrieval/BM25/data/corpus.jsonl', lines=True))
df.drop(columns=['metadata'], inplace=True)
corpus_tokens = {}

def tokenize(text):
    tokens = word_tokenize(text.lower())
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
    return filtered_tokens

for index, row in tqdm(df.iterrows(), total=df.shape[0]):
    tokens = tokenize(row['text'])
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
    corpus_tokens[row['_id']] = filtered_tokens

100%|██████████| 171332/171332 [02:34<00:00, 1108.42it/s]


In [ ]:
inverted_index = defaultdict(dict)
for doc_id, tokens in tqdm(corpus_tokens.items(), desc='Indexing...'):
    for term, frequency in Counter(tokens).items():
        inverted_index[term][doc_id] = frequency

Indexing...: 100%|██████████| 171332/171332 [00:17<00:00, 10038.64it/s]


In [ ]:
docs_len = {}
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc='Calculating doc stats...'):
    docs_len[row['_id']] = len(corpus_tokens[row['_id']])

Calculating doc stats...: 100%|██████████| 171332/171332 [00:06<00:00, 27799.29it/s]


In [ ]:
N = len(df)
average_dl = sum(docs_len.values()) / N

def bm25_score(term, doc_id, k1=0.50, b=0.75):
  if term not in inverted_index or doc_id not in inverted_index[term]:
    return 0.0

  tf = inverted_index[term][doc_id]
  dl = docs_len[doc_id]
  df = len(inverted_index[term])
  idf = math.log((N - df + 0.5) / (df + 0.5))
  denom = tf + k1 * (1 - b + b * dl / average_dl)
  score = idf * (tf * (k1 + 1) / denom)
  return score


In [ ]:
query = 'how does the coronavirus respond to changes in the weather'
query_tokens = tokenize(query)
union_docs = set().union(*(inverted_index[t].keys() for t in query_tokens))

scores = defaultdict(float)
for doc_id in tqdm(union_docs, desc='Calculating scores...'):
    score = sum(bm25_score(t, doc_id) for t in query_tokens)
    scores[doc_id] = score

sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
sorted_scores = sorted_scores[:50]
sorted_scores

Calculating scores...: 100%|██████████| 36107/36107 [00:00<00:00, 224619.32it/s]


[('aiwxlxzt', 12.584880301888276),
 ('8eitzlpf', 12.4791318647096),
 ('j6ybcv93', 11.898653253138715),
 ('h5ufxzv9', 11.70705239579898),
 ('0mikqjpj', 11.496368989904196),
 ('zvngy7zz', 11.42922713000894),
 ('124czudi', 11.331083428067622),
 ('mja93ena', 11.052538635738099),
 ('hra8otj5', 11.037711146287306),
 ('cx3lf0dx', 10.955621688089247),
 ('jc1k3fki', 10.573177505853247),
 ('39yvniki', 10.47668426487893),
 ('akb96git', 10.4666863711316),
 ('crjg72fj', 10.424918870670004),
 ('7iuw5oz8', 10.424918870670004),
 ('9se0wm4t', 10.372197475656806),
 ('ma27v97d', 10.273477435593296),
 ('k260c04b', 10.247814201415174),
 ('mmhkw2si', 10.239419818392156),
 ('lyw3pxt3', 10.2071888200686),
 ('uj8a09t3', 10.109395611653714),
 ('5ekdfers', 10.021746495323004),
 ('90p1vlkh', 9.89247726303866),
 ('ab1u2nw7', 9.89247726303866),
 ('o9oxchq6', 9.817136175464432),
 ('26gf4q1v', 9.800660175085085),
 ('yi57n8nc', 9.75846495720081),
 ('04rbtmmi', 9.752772576055328),
 ('tyhtdawb', 9.75260618093786),
 ('ei